# [SQL 재현] 소방청 화재발생 통계 — 지역유형 분석

## 단계: 05. 지역유형(군·시·구) 분석 — SQL 재현
- 목표: 시군구 이름의 끝 글자로 지역유형을 나누고, 유형별 화재 현황·발생율 중앙값·장소 구성비를 SQL로 계산해 Python 결과와 대조한다.
- 환경: Jupyter Notebook + sqlite3 (SQL_04에서 만든 analysis2, fire 테이블 사용)
- 대조 기준: nfa_fire_2024.ipynb (3.2, 3.3, 4.3 해석, 5.1, 5.4), nfa_fire_2024.sas (546~549행)
- 참고: 군위군은 경북·대구 두 행으로 나뉘어 있음 (SQL_04 4.5-4). 노트북과 대조하기 위해 두 행을 그대로 둔다.

### 5.0 환경 설정
- 새 노트북이므로 pandas·sqlite3를 불러오고 DB에 연결한다 (노트북을 새로 열거나 커널을 재시작하면 이 셀부터 다시 실행)
- 이번 노트북에서 쓸 테이블(analysis2, fire)이 DB에 있는지 확인

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'C:\data\sql_practice.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

,name
0,hira
1,hira_eda
2,hira_model
3,hira_final
4,fire_raw
5,pop
6,fire
7,fire_agg
8,analysis
9,py_result


### 5.1 지역유형 분류
#### 5.1-1 시군구 이름 끝 글자 분포 확인
- Python `str.endswith('군')`, SAS 546~549행 대응
- `SUBSTR(열, -1)`: 글자의 맨 마지막 한 글자를 꺼낸다
- 분류 규칙을 정하기 전에, 끝 글자가 실제로 어떤 것들이 있는지 먼저 확인

In [3]:
q = """
SELECT SUBSTR(SIGUNGU, -1) AS 끝글자,
       COUNT(*)            AS 지역수
FROM analysis2
GROUP BY 끝글자
ORDER BY 지역수 DESC
"""
pd.read_sql(q, conn)

,끝글자,지역수
0,구,104
1,군,83
2,시,66
3,종,1


#### 5.1-2 지역유형 2분류·3분류 열 추가 → fire_region 테이블
- 2분류(REGION_TYPE): 이름이 '군'으로 끝나면 군지역, 나머지는 시구지역 (노트북 3.2)
- 3분류(REGION_CAT): 군 / 구 / 나머지는 시 (노트북 5.4, SAS 546~549행)
- CASE WHEN은 위에서부터 차례로 조건을 보고, 처음 맞는 것 하나만 적용한다

In [7]:
conn.execute("DROP TABLE IF EXISTS fire_region")
conn.execute("""
CREATE TABLE fire_region AS
SELECT *,
        CASE WHEN SIGUNGU LIKE '%군' THEN '군지역'
            ELSE '시구지역' END        AS REGION_TYPE,
        CASE WHEN SUBSTR(SIGUNGU, -1) = '군' THEN '군'
             WHEN SUBSTR(SIGUNGU, -1) = '구' THEN '구'
             ELSE '시' END             AS REGION_CAT
FROM analysis2
""")
conn.commit()

#### 5.1-3 분류 결과 개수 확인
- 2분류와 3분류를 한 표에서 함께 확인 (GROUP BY 두 열)